In [10]:
pip install dwave-ocean-sdk

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [16]:
$dwave auth login

SyntaxError: invalid syntax (3188052668.py, line 1)

In [3]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
from dwave.system import DWaveSampler, EmbeddingComposite
from sklearn.metrics import recall_score

# Fetch the dataset
statlog_german_credit_data = fetch_ucirepo(id=144)

# Extract features and target
X = statlog_german_credit_data.data.features
y = statlog_german_credit_data.data.targets

# Handle categorical columns by encoding them
encoder = LabelEncoder()
categorical_columns = X.select_dtypes(include=["object"]).columns
for column in categorical_columns:
    X[column] = encoder.fit_transform(X[column])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Normalize numerical data (important for quantum classifiers)
numerical_columns = X.select_dtypes(include=[np.number]).columns
X_train[numerical_columns] = (X_train[numerical_columns] - X_train[numerical_columns].mean()) / X_train[numerical_columns].std()
X_test[numerical_columns] = (X_test[numerical_columns] - X_test[numerical_columns].mean()) / X_test[numerical_columns].std()

# Output class distribution
print(f"Class distribution in y_train: {y_train.value_counts()}")

# Define the cost matrix (as per the problem description)
# 1 = Good, 2 = Bad
cost_matrix = np.array([[0, 1],  # Misclassifying Good as Bad incurs penalty 1
                       [5, 0]]) # Misclassifying Bad as Good incurs penalty 5

# Function to prepare QUBO matrix with cost matrix incorporation
def prepare_qubo_with_cost_matrix(X, y, cost_matrix):
    num_samples, num_features = X.shape
    
    # Initialize the QUBO matrix
    Q = np.zeros((num_features, num_features))
    
    # For every feature pair, calculate interaction and add it to QUBO
    for i in range(num_features):
        for j in range(num_features):
            # Calculate pairwise interaction between features
            Q[i, j] = np.dot(X.iloc[:, i], X.iloc[:, j])  
    
    # Apply cost matrix penalty to the QUBO for misclassifications
    for idx, label in enumerate(y):
        if label == 2:  # "Bad" credit class (minority class)
            for i in range(num_features):
                Q[i, i] += cost_matrix[1][0]  # Misclassifying Bad as Good (false negative)
        else:  # "Good" credit class
            for i in range(num_features):
                Q[i, i] += cost_matrix[0][1]  # Misclassifying Good as Bad (false positive)
    
    return Q

# Prepare QUBO problem for training data
Q = prepare_qubo_with_cost_matrix(X_train, y_train, cost_matrix)

# Setup the D-Wave sampler (quantum solver)
sampler = EmbeddingComposite(DWaveSampler())

# Solve the QUBO problem using the quantum sampler
response = sampler.sample_qubo(Q, num_reads=1000)

# Extract the best solution from the response
solution = response.first.sample

# Print the best solution found by the quantum solver
print("Best solution from quantum solver:")
print(solution)

# Map the solution to predicted labels (1 for "Good" credit, 2 for "Bad" credit)
y_pred = [1 if solution[i] == 0 else 2 for i in range(len(X_test))]

# Evaluate recall for the minority class (Bad credit)
recall = recall_score(y_test, y_pred, pos_label=2)

# Print the recall score for the minority class
print(f"Recall for the minority class (Bad credit): {recall:.3f}")



Class distribution in y_train: class
1        491
2        209
Name: count, dtype: int64


/var/folders/v4/gng44qs519q3hzsgrnlm91d40000gn/T/ipykernel_1971/210999970.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[column] = encoder.fit_transform(X[column])
/var/folders/v4/gng44qs519q3hzsgrnlm91d40000gn/T/ipykernel_1971/210999970.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[column] = encoder.fit_transform(X[column])
/var/folders/v4/gng44qs519q3hzsgrnlm91d40000gn/T/ipykernel_1971/210999970.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFr

ValueError: API token not defined